# ECON417 — Project 1: Healthcare Analytics
## Predicting Patient Disease Progression for Government Health Programs

**Domain:** Government Healthcare
**Dataset:** Diabetes Clinical Dataset (Efron et al., 2004)
**Research Question:** Which patient health metrics best predict disease progression?

---

### Learning Objectives
1. Apply demeaning (mean-centering) and understand its statistical interpretation
2. Implement and compare OLS, Ridge, and Lasso regression
3. Build a Random Forest model and extract feature importance
4. Use R², RMSE, and MAE to evaluate and compare model performance

### Background

The **Centers for Medicare & Medicaid Services (CMS)** — the largest federal health insurer in the U.S. —
routinely builds predictive models from clinical data to:
- Identify high-risk patients for proactive intervention
- Allocate resources across Medicare and Medicaid programs
- Evaluate the effectiveness of disease management initiatives
- Estimate future healthcare cost burdens

This project uses a real clinical dataset of **442 diabetes patients** collected at Stanford.
The target variable is a **quantitative measure of disease progression one year after baseline** —
a direct proxy for patient outcomes and downstream healthcare costs.

| Feature | Description |
|---------|-------------|
| `age` | Patient age |
| `sex` | Patient sex |
| `bmi` | Body Mass Index |
| `bp` | Average blood pressure |
| `s1` | Total serum cholesterol |
| `s2` | Low-density lipoprotein (LDL) |
| `s3` | High-density lipoprotein (HDL) |
| `s4` | Total cholesterol / HDL ratio |
| `s5` | Log of serum triglycerides level |
| `s6` | Blood glucose level |
| **`progression`** | **Disease progression score at 1 year (target)** |


## Section 0 — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, RidgeCV, LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
sns.set_theme(style="whitegrid")
print("All libraries loaded successfully!")

## Section 1 — Data Acquisition

### Data Sources

**This project (sklearn built-in):**
```python
from sklearn.datasets import load_diabetes
```

**Where government analysts get similar data in practice:**
- **CMS Open Data Portal:** https://data.cms.gov/ — Hospital Compare, Medicare Claims, HEDIS measures
- **CDC NHANES:** https://www.cdc.gov/nchs/nhanes/ — National Health and Nutrition Examination Survey
- **NIH dbGaP:** https://www.ncbi.nlm.nih.gov/gap/ — Genomics and phenotype data

> The sklearn diabetes dataset originates from a real Stanford clinical study and is
> structurally identical to the patient-level data used in CMS risk stratification models.

In [ ]:
from sklearn.datasets import load_diabetes

diabetes = load_diabetes()

feature_names = ['age', 'sex', 'bmi', 'bp',
                 's1_cholesterol', 's2_ldl', 's3_hdl',
                 's4_tch', 's5_triglycerides', 's6_glucose']

X = pd.DataFrame(diabetes.data, columns=feature_names)
y = pd.Series(diabetes.target, name='progression')

print("Dataset Summary:")
print(f"  Samples  : {X.shape[0]}")
print(f"  Features : {X.shape[1]}")
print(f"  Target range: {y.min():.1f}  to  {y.max():.1f}")
print(f"  Target mean : {y.mean():.2f}  |  std: {y.std():.2f}")
X.head()

## Section 2 — Exploratory Data Analysis

> **Principle:** Always understand your data before modeling.
> In clinical settings, outliers may carry real medical significance — never remove them without justification.

In [ ]:
print("Descriptive Statistics:")
X.describe().round(3)

In [ ]:
# Distribution of the target variable
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(y, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(y.mean(),   color='red',    linestyle='--', lw=2, label=f'Mean   = {y.mean():.1f}')
axes[0].axvline(y.median(), color='orange', linestyle='--', lw=2, label=f'Median = {y.median():.1f}')
axes[0].set_xlabel('Disease Progression Score (1-year)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Disease Progression Score')
axes[0].legend()

axes[1].boxplot(y, vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6),
                medianprops=dict(color='red', lw=2))
axes[1].set_ylabel('Disease Progression Score')
axes[1].set_title('Boxplot — Disease Progression Score')

plt.tight_layout()
plt.show()
print(f"Skewness: {y.skew():.3f}  |  Kurtosis: {y.kurt():.3f}")

In [ ]:
# Correlation heatmap
data_full = X.copy()
data_full['progression'] = y
corr = data_full.corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix (including Target)', fontsize=13)
plt.tight_layout()
plt.show()

print("Correlation with target (sorted by absolute value):")
corrs = corr['progression'].drop('progression').abs().sort_values(ascending=False)
for feat, val in corrs.items():
    bar = '█' * int(val * 25)
    print(f"  {feat:20s}: {val:.3f}  {bar}")

In [ ]:
# Scatter plots: top 4 most correlated features vs target
top4 = corrs.index[:4].tolist()

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, feat in enumerate(top4):
    axes[i].scatter(X[feat], y, alpha=0.4, color='steelblue', s=22,
                    edgecolors='white', linewidths=0.3)
    z = np.polyfit(X[feat], y, 1)
    x_line = np.linspace(X[feat].min(), X[feat].max(), 100)
    axes[i].plot(x_line, np.poly1d(z)(x_line), 'r-', lw=2.5,
                 label=f'r = {corrs[feat]:.2f}')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Progression')
    axes[i].set_title(f'{feat} vs Target')
    axes[i].legend(fontsize=9)
plt.suptitle('Top 4 Correlated Features vs Disease Progression', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Section 3 — Demeaning (Mean-Centering)

### Definition

$$X_{\text{demeaned}} = X - \bar{X}$$

Subtract each feature's sample mean so that every feature has mean = 0.

### Why Demean?

| Reason | Explanation |
|--------|-------------|
| **Intercept interpretation** | After demeaning, $\beta_0$ equals the mean of $y$ — directly interpretable |
| **Numerical stability** | Reduces floating-point rounding errors in matrix inversion |
| **Prerequisite for regularization** | Lasso and Ridge assume features are centered before the penalty is applied |
| **Clinical meaning** | Coefficients represent the effect of being *above average* on that metric |

> **Note — Demeaning vs. Standardization:**
> - **Demeaning:** subtract the mean only (preserves original units/scale)
> - **Standardization (Z-score):** subtract mean *and* divide by std (scales to unit variance)
>
> We will demean for OLS, then additionally standardize for Lasso/Ridge
> because regularization is sensitive to feature scale.

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Compute X_mean = X.mean(), then X_demeaned = X - X_mean. Verify by printing X_demeaned.mean() — values should be ≈ 0.
raise NotImplementedError()

## Section 4 — Train / Test Split

- **Training set (80%):** used to fit all models
- **Test set (20%):** held out to evaluate generalization

> **Clinical note:** In real medical research, splits are typically done by time (to prevent
> future data leakage) or by patient ID (to prevent the same patient appearing in both sets).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_demeaned, y, test_size=0.2, random_state=42
)

print(f"Training set : {X_train.shape[0]} patients")
print(f"Test set     : {X_test.shape[0]} patients")
print(f"\nTrain target mean : {y_train.mean():.2f}")
print(f"Test  target mean : {y_test.mean():.2f}")

# Additional standardization for Lasso and Ridge (fit on train only!)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print("\nStandardized features ready for Lasso and Ridge.")

## Section 5 — Evaluation Metrics

We will use three metrics throughout this project:

| Metric | Formula | Interpretation |
|--------|---------|----------------|
| **R²** | $1 - SS_{res}/SS_{tot}$ | Fraction of variance explained; 1 = perfect, 0 = baseline mean |
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ | Average error in the **same units as y**; penalizes large errors |
| **MAE** | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Average absolute error; more **robust to outliers** than RMSE |

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Define evaluate_model(y_true, y_pred, model_name) that computes and returns R2, RMSE, and MAE using r2_score(), mean_squared_error(), mean_absolute_error()
raise NotImplementedError()

## Section 6 — Ordinary Least Squares (OLS) Regression

OLS minimizes the sum of squared residuals:

$$\hat{\beta}_{OLS} = \underset{\beta}{\arg\min} \|y - X\beta\|_2^2 = (X^\top X)^{-1} X^\top y$$

This is our **baseline model**. Because we demeaned $X$, the intercept $\hat{\beta}_0$
will equal the training-set mean of $y$.

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Create a LinearRegression() object, fit on (X_train, y_train), predict on X_test, and call evaluate_model(). Store the result dict in lr_res.
raise NotImplementedError()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Predicted vs Actual
axes[0].scatter(y_test, y_pred_lr, alpha=0.5, color='steelblue', s=35,
                edgecolors='white', linewidths=0.3)
lims = [min(y_test.min(), y_pred_lr.min()), max(y_test.max(), y_pred_lr.max())]
axes[0].plot(lims, lims, 'r--', lw=2, label='Perfect fit')
axes[0].set_xlabel('Actual Progression')
axes[0].set_ylabel('Predicted Progression')
axes[0].set_title('OLS — Predicted vs Actual')
axes[0].legend()

# Residual plot
resid = y_test.values - y_pred_lr
axes[1].scatter(y_pred_lr, resid, alpha=0.5, color='teal', s=35,
                edgecolors='white', linewidths=0.3)
axes[1].axhline(0, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Values')
axes[1].set_ylabel('Residuals  (y − ŷ)')
axes[1].set_title('OLS — Residual Plot')

# Coefficient bar chart
c_colors = ['steelblue' if c > 0 else 'coral' for c in lr.coef_]
axes[2].barh(X.columns, lr.coef_, color=c_colors, edgecolor='white', alpha=0.85)
axes[2].axvline(0, color='black', lw=1)
axes[2].set_xlabel('Coefficient Value')
axes[2].set_title('OLS — Regression Coefficients')

plt.tight_layout()
plt.show()

## Section 7 — Ridge Regression (L2 Regularization)

Ridge adds an L2 penalty — the sum of *squared* coefficients — to the OLS objective:

$$\hat{\beta}_{Ridge} = \underset{\beta}{\arg\min} \|y - X\beta\|_2^2 + \alpha \|\beta\|_2^2$$

- **Effect:** All coefficients are *shrunk toward zero*, but none become exactly zero
- **Best for:** Multicollinear features (e.g., the blood lipid panel — s1 through s4 are correlated)
- **Hyperparameter α:** Larger α → stronger shrinkage. We select it via 5-fold cross-validation.

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Use RidgeCV(alphas=np.logspace(-3, 3, 100), cv=5) to find the best alpha on (X_train_scaled, y_train). Then fit Ridge(alpha=best_alpha_ridge) and predict on X_test_scaled. Call evaluate_model() and store the result in ridge_res.
raise NotImplementedError()

In [ ]:
# Ridge coefficient path — how each coefficient changes as alpha varies
alpha_range = np.logspace(-3, 3, 60)
paths_ridge = np.array([
    Ridge(alpha=a).fit(X_train_scaled, y_train).coef_ for a in alpha_range
])

plt.figure(figsize=(11, 6))
for i, col in enumerate(X.columns):
    plt.semilogx(alpha_range, paths_ridge[:, i], lw=2, label=col)
plt.axvline(best_alpha_ridge, color='black', linestyle='--', lw=2.5,
            label=f'Best α = {best_alpha_ridge:.3f}')
plt.xlabel('Regularization Strength α  (log scale)')
plt.ylabel('Coefficient Value')
plt.title('Ridge Regression — Coefficient Path (L2)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

## Section 8 — Lasso Regression (L1 Regularization)

Lasso adds an L1 penalty — the sum of *absolute* coefficients — to the OLS objective:

$$\hat{\beta}_{Lasso} = \underset{\beta}{\arg\min} \|y - X\beta\|_2^2 + \alpha \|\beta\|_1$$

- **Key property:** The L1 penalty drives some coefficients to **exactly zero**, performing
  automatic feature selection
- **Best for:** High-dimensional data where many features may be irrelevant
  (e.g., identifying which biomarkers actually predict outcomes)
- **Contrast with Ridge:** Ridge *shrinks* all features; Lasso *eliminates* some

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Use LassoCV(alphas=np.logspace(-3, 3, 100), cv=5, max_iter=10000) to find the best alpha. Then fit Lasso(alpha=best_alpha_lasso, max_iter=10000) and predict. Call evaluate_model(), store in lasso_res. Print which features were zeroed out.
raise NotImplementedError()

In [ ]:
# Lasso coefficient path — notice coefficients hitting zero as alpha increases
paths_lasso = np.array([
    Lasso(alpha=a, max_iter=10000).fit(X_train_scaled, y_train).coef_
    for a in alpha_range
])

plt.figure(figsize=(11, 6))
for i, col in enumerate(X.columns):
    plt.semilogx(alpha_range, paths_lasso[:, i], lw=2, label=col)
plt.axvline(best_alpha_lasso, color='black', linestyle='--', lw=2.5,
            label=f'Best α = {best_alpha_lasso:.3f}')
plt.xlabel('Regularization Strength α  (log scale)')
plt.ylabel('Coefficient Value')
plt.title('Lasso Regression — Coefficient Path (L1)  [Notice the zeros!]')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.show()

## Section 9 — Comparing the Three Linear Models

| Metric | Direction | What it Measures |
|--------|-----------|-----------------|
| **R²** | ↑ higher is better | Fraction of variance explained |
| **RMSE** | ↓ lower is better | Average error (same units as y) |
| **MAE** | ↓ lower is better | Average absolute error (outlier-robust) |

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Build a DataFrame from [lr_res, ridge_res, lasso_res], set 'Model' as the index, round to 4 decimal places, and print the comparison table.
raise NotImplementedError()

In [ ]:
# Side-by-side coefficient comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
coef_sets = [
    (lr.coef_,    'OLS Coefficients',   'steelblue'),
    (ridge.coef_, 'Ridge Coefficients', 'teal'),
    (lasso.coef_, 'Lasso Coefficients', 'coral'),
]
for ax, (coef, title, color) in zip(axes, coef_sets):
    c_colors = [color if c >= 0 else '#c62828' for c in coef]
    ax.barh(X.columns, coef, color=c_colors, edgecolor='white', alpha=0.87)
    ax.axvline(0, color='black', lw=1)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Coefficient Value')
plt.suptitle('Coefficient Comparison: OLS vs Ridge vs Lasso', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print("Observation: Lasso zeroes out some features (automatic selection).")
print("Ridge shrinks all features but retains them.")

## Section 10 — Random Forest

Random Forest is an ensemble of decision trees trained via bootstrap aggregation (bagging)
with random feature subsets at each split:

$$\hat{y} = \frac{1}{B} \sum_{b=1}^{B} T_b(x)$$

### Why use Random Forest after linear models?

| Property | Linear Models | Random Forest |
|----------|--------------|---------------|
| Captures nonlinearity | ✗ | ✓ |
| Feature interactions | Limited | ✓ Automatic |
| Robust to outliers | Moderate | ✓ |
| Requires standardization | Yes (Ridge/Lasso) | ✗ |
| Interpretability | High (coefficients) | Moderate (feature importance) |

### Key Hyperparameters
- `n_estimators` — number of trees (more = more stable, slower)
- `max_depth` — maximum depth per tree (controls overfitting)
- `min_samples_leaf` — minimum samples at a leaf node (smoothing)

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Create RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=5, random_state=42, n_jobs=-1), fit on (X_train, y_train), predict on X_test, call evaluate_model(), store in rf_res.
raise NotImplementedError()

In [ ]:
### ===== YOUR CODE HERE =====
# Hint: Extract rf.feature_importances_, build a DataFrame with columns ['Feature', 'Importance'], sort by Importance descending, store as importance_df.
raise NotImplementedError()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Feature importance bar chart
palette_rf = sns.color_palette("Blues_r", len(importance_df))
axes[0].barh(importance_df['Feature'][::-1],
             importance_df['Importance'][::-1],
             color=palette_rf, edgecolor='white', alpha=0.9)
for i, (_, row) in enumerate(importance_df[::-1].iterrows()):
    axes[0].text(row['Importance'] + 0.002, i,
                 f"{row['Importance']:.3f}", va='center', fontsize=9)
axes[0].set_xlabel('Feature Importance (Mean Decrease Impurity)')
axes[0].set_title('Random Forest — Feature Importance')

# Predicted vs Actual
axes[1].scatter(y_test, y_pred_rf, alpha=0.5, color='teal', s=35,
                edgecolors='white', linewidths=0.3)
lims = [min(y_test.min(), y_pred_rf.min()), max(y_test.max(), y_pred_rf.max())]
axes[1].plot(lims, lims, 'r--', lw=2, label='Perfect fit')
axes[1].set_xlabel('Actual Disease Progression')
axes[1].set_ylabel('Predicted Disease Progression')
axes[1].set_title('Random Forest — Predicted vs Actual')
r2_rf = r2_score(y_test, y_pred_rf)
axes[1].text(0.05, 0.95, f'R² = {r2_rf:.4f}', transform=axes[1].transAxes,
             fontsize=12, va='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[1].legend()

plt.tight_layout()
plt.show()

## Section 11 — Final Model Comparison and Conclusion

In [ ]:
results_final = pd.DataFrame(results_all).set_index('Model').round(4)
print("All Models — Final Performance Summary:")
print("="*58)
print(results_final.to_string())
print("="*58)
print(f"\nBest R²  : {results_final['R2'].idxmax()}   ({results_final['R2'].max():.4f})")
print(f"Best RMSE: {results_final['RMSE'].idxmin()}   ({results_final['RMSE'].min():.4f})")
print(f"Best MAE : {results_final['MAE'].idxmin()}   ({results_final['MAE'].min():.4f})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 6))
palette = ['#1565C0', '#00695C', '#BF360C', '#2E7D32']
for i, metric in enumerate(['R2', 'RMSE', 'MAE']):
    vals = results_final[metric]
    bars = axes[i].bar(vals.index, vals, color=palette[:len(vals)],
                       edgecolor='white', alpha=0.88)
    for bar, val in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2.,
                     bar.get_height() + max(vals)*0.01,
                     f'{val:.3f}', ha='center', va='bottom',
                     fontsize=9, fontweight='bold')
    title_map = {'R2': 'R²', 'RMSE': 'RMSE', 'MAE': 'MAE'}
    axes[i].set_title(title_map[metric], fontsize=12)
    axes[i].set_ylabel(title_map[metric])
    axes[i].tick_params(axis='x', rotation=20)
    if metric == 'R2':
        axes[i].set_ylim(0, 1.12)
plt.suptitle('Healthcare Project — All Models Performance Comparison',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Conclusions and Policy Implications

### Model Performance Takeaways

**OLS (Baseline):** Transparent and interpretable — every coefficient directly quantifies a
feature's linear contribution to disease progression. Best choice when you need to
report results to non-technical policymakers.

**Ridge:** More stable than OLS when features are correlated (the blood lipid panel is
a classic example of multicollinearity). Retains all features with shrunk coefficients.

**Lasso:** Performs automatic feature selection — only the clinically relevant biomarkers
remain. Useful for building parsimonious screening tools.

**Random Forest:** Typically the highest predictive accuracy. Captures nonlinear
relationships (e.g., threshold effects of BMI or glucose on outcomes) and feature
interactions that linear models miss.

### Key Clinical Findings

Based on feature importance (Random Forest) and coefficient magnitude (OLS/Lasso):
- **BMI** and **blood glucose (s6)** are the strongest predictors
- These align with clinical guidelines: BMI and glycemic control are the two most
  actionable targets in diabetes management programs

### Recommendations for a CMS Analyst

1. Deploy the Random Forest for *risk stratification* — flag patients with predicted
   scores in the top quartile for proactive case management
2. Use Lasso coefficients for *reporting to Congress* — the sparse model is easier
   to explain and audit
3. Retrain models quarterly as new claims data becomes available
4. Conduct a **fairness audit** — evaluate model performance across race, sex, and
   income subgroups before production deployment

### Discussion Questions

> 1. OLS, Ridge, and Lasso all assume a linear relationship between features and the target.
>    What type of feature transformation could you add to capture nonlinearity
>    *without* switching to Random Forest?
>
> 2. Random Forest does not produce a simple formula. How would you explain its
>    predictions to a non-technical official at HHS?
>
> 3. What are the ethical risks of using a predictive model to allocate healthcare resources?
>    Which fairness criterion (demographic parity, equalized odds, etc.) would you apply here?